In [1]:
# 1. Libraries
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

nltk.download('stopwords')
nltk.download('wordnet')

# 2. Load dataset
df = pd.read_csv("community_datasets.csv")
print(df.head())
print(df.shape)

# 3. Select text column
text_column = "description"   # CHANGE THIS if needed
df[text_column] = df[text_column].fillna("").astype(str)

# 4. NLP cleaning
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words
             if w not in stop_words and len(w) > 2]
    return " ".join(words)

df["clean_text"] = df[text_column].apply(clean_text)

# 5. TF-IDF
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X = vectorizer.fit_transform(df["clean_text"])

# 6. Find K using elbow method
inertia = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)
    inertia.append(model.inertia_)

plt.plot(range(2,9), inertia, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

# 7. K-Means
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# 8. Show cluster sizes
print(df["cluster"].value_counts().sort_index())

# 9. Show important words in each cluster
terms = vectorizer.get_feature_names_out()

for i in range(k):
    words = kmeans.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:")
    print(", ".join(terms[words]))

# 10. Visualise clusters
svd = TruncatedSVD(n_components=2, random_state=42)
X2 = svd.fit_transform(X)

plt.figure(figsize=(8,6))
plt.scatter(X2[:,0], X2[:,1], c=df["cluster"])
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Community Dataset - K-Means Clustering")
plt.show()

# 11. Save results
df.to_csv("community_datasets_clustered.csv", index=False)

print("Done! Results saved.")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


                                                  id  \
0  projects/sat-io/open-datasets/shoreline/mainlands   
1  projects/sat-io/open-datasets/shoreline/big_is...   
2  projects/sat-io/open-datasets/shoreline/small_...   
3         projects/sat-io/open-datasets/S2COAST-2023   
4       projects/sat-io/open-datasets/OSM_waterLayer   

                                            provider  \
0              United States Geological Survey, USGS   
1              United States Geological Survey, USGS   
2              United States Geological Survey, USGS   
3  Yuanqiang Duan, Arturo Sanchez-Azofeifa, Chunp...   
4  Institute of Industrial Sciences, Open Street ...   

                                               title              type  \
0                           Global Shoreline Dataset             table   
1                           Global Shoreline Dataset             table   
2             Global Shoreline Dataset Small Islands             table   
3  S2Coast-2023 Global 10-mete

KeyError: 'description'